# Agentic RAG: Router–Retriever System
# Author: Parthasarathy
# Project: Multi-Agent Orchestration with CrewAI
# Module Install: Install Crewai as per requirement, littlellm for pdf search

In [2]:
%pip install -U \
    crewai \
    crewai-tools \
    langchain \
    langchain-community \
    langchain-openai \
    langchain-text-splitters \
    faiss-cpu \
    pypdf \
    python-dotenv \
    requests \
    pandas \
    matplotlib \
    ipywidgets \
    openai \
    python-dotenv

  Using cached crewai-1.15.22-py3-none-any.whl (1.2 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 841.4/841.4 KB 3.9 MB/s eta 0:00:0000:0100:01
  Using cached langchain-1.4.2-py3-none-any.whl (163 kB)
  Using cached faiss_cpu-1.15.1-cp310-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (18.8 MB)
  Using cached pypdf-6.19.0-py3-none-any.whl (395 kB)
  Using cached pandas-2.3.3-cp310-cp310-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (12.8 MB)
  Using cached matplotlib-3.10.9-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (8.8 MB)
  Using cached ipywidgets-8.1.9-py3-none-any.whl (140 kB)
  Using cached openai-3.16.2-py3-none-any.whl (2.1 MB)
  Using cached crewai_core-1.15.22-py3-none-any.whl (44 kB)
  Using cached crewai_cli-1.15.22-py3-none-any.whl (202 kB)
  Using cached pytz-2026.3.post1-py2.py3-none-any.whl (508 kB)
  Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
  Using cached fonttools-4.65.0-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl 

In [3]:
# -------------------------------
# Import Modules
# -------------------------------
import json
import os
import re
import time
from pathlib import Path
from typing import Type

import matplotlib.pyplot as plt
import pandas as pd
import requests
from dotenv import load_dotenv
from pydantic import BaseModel, Field

from crewai import Agent, Crew, LLM, Process, Task
from crewai.tools import BaseTool
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from crewai_tools import TavilySearchTool
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings

/tmp/ipykernel_6673/873673663.py:20: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [4]:
import os
from dotenv import load_dotenv

load_dotenv()

embedding_model = OpenAIEmbeddings(
    model=os.getenv(
        "OPENAI_EMBEDDING_MODEL",
        "text-embedding-3-small",
    ),
    api_key=os.getenv("OPENAI_API_KEY"),
)
try:
    embedding = embedding_model.embed_query("Test connection")
    print("Embedding connection successful.")
    print("Embedding dimensions:", len(embedding))
except Exception as error:
    print(type(error).__name__, error)


Embedding connection successful.
Embedding dimensions: 1536


In [ ]:
from crewai_tools import PDFSearchTool

pdf_paths = ["trasformer_research_paper-dataset.pdf"]

documents = []
for path in pdf_paths:
    loader = PyPDFLoader(path)
    documents.extend(loader.load())

# ✅ Split into manageable chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
chunks = splitter.split_documents(documents)

#Tools
pdf_tool = PDFSearchTool(file_path="trasformer_research_paper-dataset.pdf")
#pdf_tool = PDFSearchTool(documents)
web_tool = TavilySearchTool()

#print(pdf_tool.)
# Use them
try:
    pdf_result = pdf_tool.run("Explain scaled dot-product attention")
    print("PDF search result:\n", pdf_result)
except Exception as exc:
    print(f"PDF search failed: {exc}")
print(web_tool.run("Latest applications of Transformers in 2026"))

PDF search result:
 Relevant Content:
No relevant content found.
{
  "query": "Latest applications of Transformers in 2026",
  "follow_up_questions": null,
  "answer": null,
  "images": [],
  "results": [
    {
      "url": "https://uk.finance.yahoo.com/news/transformers-industry-research-2026-2036-112300793.html",
      "title": "Transformers Industry Research 2026-2036 - Advanced Power Equipment, High-Speed Signal Demand and 5G-Enabled Smart Grid Infrastructure Create New Opportunities",
      "content": "Dublin, May 06, 2026 (GLOBE NEWSWIRE) -- The \"Transformers Market by Type (Power, Distribution, Instrument, Special Application), Cooling Method (Liquid-immersed, Dry-type), and Application (Utilities, Industrial, Renewables, Railways) - Global Forecast to 2036\" report has been added to ResearchAndMarkets.com's offering. [...] The global transformers market is projected to skyrocket to USD 113.0 billion by 2036, advancing from USD 62.2 billion in 2026, with a robust CAGR of 6.2%. 

In [7]:
# --------------------------------------------------
# Hardcoded credentials
# --------------------------------------------------
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=os.getenv("OPENAI_CHAT_MODEL", "gpt-4o-mini"),
    temperature=0,
    api_key=os.getenv("OPENAI_API_KEY"),
)

In [8]:

# -------------------------------
# Define Agents
# -------------------------------

# Router Agent: decides retrieval path
router_agent = Agent(
    role="Router Agent",
    goal="Classify user questions and decide retrieval path (PDF, Web, or LLM).",
    backstory="Expert in query classification and routing for multi-source retrieval.",
    tools=[pdf_tool, web_tool],
    model="gpt-5.6-sol",
    verbose=True
)

# Retriever Agent: executes retrieval
retriever_agent = Agent(
    role="Retriever Agent",
    goal="Retrieve answers from the chosen source and ground them in evidence.",
    backstory="Specialist in executing retrieval from static PDFs or dynamic web sources.",
    tools=[pdf_tool, web_tool],
    model="gpt-5.6-sol",
    verbose=True
)

# -------------------------------
# Define Tasks
# -------------------------------

# Task 1: Router decides retrieval path
router_task = Task(
    description=(
        "Analyze the user question. "
        "If it relates to Transformer research paper, route to PDFSearchTool. "
        "If it requires fresh or external info, route to TavilySearchResults. "
        "Otherwise, use LLM directly."
    ),
    agent=router_agent,
    expected_output="Decision: {PDF | Web | LLM}"
)

# Task 2: Retriever executes retrieval
retriever_task = Task(
    description=(
        "Based on Router decision, execute retrieval. "
        "Use PDFSearchTool for static content, TavilySearchResults for dynamic info. "
        "Return grounded answer with citations."
    ),
    agent=retriever_agent,
    expected_output="Grounded answer with source attribution."
)

# -------------------------------
# Orchestration
# -------------------------------
# For a crew
#await crewai.kickoff_async()
#agent.kickoff_async()
# Crew kickoff is performed in run_demo() after crew is created.

crew = Crew(
    agents=[router_agent, retriever_agent],
    tasks=[router_task, retriever_task],
    tracing=True,
    verbose=True
)

# -------------------------------
# Demo Run
# -------------------------------
async def run_demo(question: str):
    print(f"\nUser Question: {question}\n")
    result = await crew.kickoff_async(inputs={"question": question})
    print("\n--- Final Answer ---")
    print(result)

# -------------------------------
# Example Queries
# -------------------------------
await run_demo("Explain scaled dot-product attention in the Transformer.")

await run_demo("What are the latest applications of Transformers in 2026?")



User Question: Explain scaled dot-product attention in the Transformer.



╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: fe7907f9-5224-4dc8-bac5-23d48163800c                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analyze the user question. If it relates to Transformer research paper, route to PDFSearchTool. If it    │
│  requires fresh or external info, route to TavilySearchResults. Otherwise, use LLM directly.                    │
│  ID: 62223ae3-69c4-486f-a1d9-efd69286b89f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Router Agent                                                                                            │
│                                                                                                                 │
│  Task: Analyze the user question. If it relates to Transformer research paper, route to PDFSearchTool. If it    │
│  requires fresh or external info, route to TavilySearchResults. Otherwise, use LLM directly.                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Retriever Agent                                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Here is the complete content related to "Router decision in computer networks" retrieved from the sources:     │
│                                                                                                                 │
│  1. From Medium                                                                                                 │
│  (https://afzalbadshah.medium.com/understanding-routing-in-computer-networks-from-basics-to-advanced-protocols  │
│  -7f3542fe695f):                                                                                                │
│  "Think about how you use Google Maps or a delivery app every day. When you request directions, it doesn’t      │
│  direct you blindly; it calculates the best route considering roads, traffic, and time. Similarly, in computer  │
│  networks, packets need smart decisions on where to go and how to get there efficiently. Just like your map     │
│  app reroutes when a road closes, routers dynamically reroute data to keep it moving smoothly. The complete     │
│  tutorial on Computer Networks can be explored here. [...] for data to travel efficiently and reach the         │
│  correct destination. It ensures that information reaches its destination efficiently and accurately. Routers   │
│  are the key devices responsible for making routing decisions in a network. [...] Routing protocols are         │
│  fundamental to network communication, and two prominent categories are Distance Vector and Link State          │
│  protocols. Distance Vector protocols, exemplified by Routing Information Protocol (RIP), make routing          │
│  decisions based on cumulative distanc..."                                                                      │
│                                                                                                                 │
│  2. From YouTube video titled "What is a Router | How Routing Actually Works in Computer Networks | Uplatz"     │
│  (https://www.youtube.com/watch?v=aXstcuc0qb0):                                                                 │
│  "A Router is a networking device responsible for connecting different networks and forwarding data packets     │
│  from one network to another based on destination IP addresses.                                                 │
│                                                                                                                 │
│  When a device sends data to another network, the router examines the packet header, checks its routing table,  │
│  determines the most efficient path available, and forwards the packet toward its destination. [...] Every      │
│  time you open a website, send a message, access cloud applications, stream a video, or connect to enterprise   │
│  infrastructure, data must travel across multiple networks before reaching its destination. The devices         │
│  responsible for deciding where that data should go and which path it should take are called Routers.           │
│                                                                                                                 │
│  In this video, you will learn: [...] [2:45] Calculating a static end-to-end path is mathematically             │
│  impossible. Instead, routers use next hop logic, calculating the optimal path only to the most [2:54]          │
│  immediate adjacent node. The receiving router performs

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Based on Router decision, execute retrieval. Use PDFSearchTool for static content, TavilySearchResults   │
│  for dynamic info. Return grounded answer with citations.                                                       │
│  Agent: Retriever Agent                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: fe7907f9-5224-4dc8-bac5-23d48163800c                                                                       │
│  Final Output: Here is the complete content related to "Router decision in computer networks" retrieved from    │
│  the sources:                                                                                                   │
│                                                                                                                 │
│  1. From Medium                                                                                                 │
│  (https://afzalbadshah.medium.com/understanding-routing-in-computer-networks-from-basics-to-advanced-protocols  │
│  -7f3542fe695f):                                                                                                │
│  "Think about how you use Google Maps or a delivery app every day. When you request directions, it doesn’t      │
│  direct you blindly; it calculates the best route considering roads, traffic, and time. Similarly, in computer  │
│  networks, packets need smart decisions on where to go and how to get there efficiently. Just like your map     │
│  app reroutes when a road closes, routers dynamically reroute data to keep it moving smoothly. The complete     │
│  tutorial on Computer Networks can be explored here. [...] for data to travel efficiently and reach the         │
│  correct destination. It ensures that information reaches its destination efficiently and accurately. Routers   │
│  are the key devices responsible for making routing decisions in a network. [...] Routing protocols are         │
│  fundamental to network communication, and two prominent categories are Distance Vector and Link State          │
│  protocols. Distance Vector protocols, exemplified by Routing Information Protocol (RIP), make routing          │
│  decisions based on cumulative distanc..."                                                                      │
│                                                                                                                 │
│  2. From YouTube video titled "What is a Router | How Routing Actually Works in Computer Networks | Uplatz"     │
│  (https://www.youtube.com/watch?v=aXstcuc0qb0):                                                                 │
│  "A Router is a networking device responsible for connecting different networks and forwarding data packets     │
│  from one network to another based on destination IP addresses.                                                 │
│                                                                                                                 │
│  When a device sends data to another network, the router examines the packet header, checks its routing table,  │
│  determines the most efficient path available, and forwards the packet toward its destination. [...] Every      │
│  time you open a website, send a message, access cloud applications, stream a video, or connect to enterprise   │
│  infrastructure, data must travel across multiple networks before reaching its destination. The devices         │
│  responsible for deciding where that data should go and which path it should take are called Routers.           │
│                                                                                                                 │
│  In this video, you will learn: [...] [2:45] Calculating a static end-to-end path is mathematically             │
│  impossible. Instead, routers use next hop logic, calc


--- Final Answer ---
Here is the complete content related to "Router decision in computer networks" retrieved from the sources:

1. From Medium (https://afzalbadshah.medium.com/understanding-routing-in-computer-networks-from-basics-to-advanced-protocols-7f3542fe695f):
"Think about how you use Google Maps or a delivery app every day. When you request directions, it doesn’t direct you blindly; it calculates the best route considering roads, traffic, and time. Similarly, in computer networks, packets need smart decisions on where to go and how to get there efficiently. Just like your map app reroutes when a road closes, routers dynamically reroute data to keep it moving smoothly. The complete tutorial on Computer Networks can be explored here. [...] for data to travel efficiently and reach the correct destination. It ensures that information reaches its destination efficiently and accurately. Routers are the key devices responsible for making routing decisions in a network. [...] Routing